In [34]:
!pip install -q langchain_docling langchain_huggingface langchain_qdrant langchain_groq langchain_community docling

## Step 1 and Step 2 - Loading the file and Chunk using Hybrid Chunker using langchain docling loader

In [35]:
from langchain_docling import DoclingLoader
from docling.chunking  import HybridChunker

SOURCE = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/AtliqAI_HR_Policies.pdf"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
loader = DoclingLoader(
    file_path=SOURCE,
    chunker=HybridChunker(tokenizer=EMBEDDING_MODEL),
)
docs = loader.load()  # loads + chunks in one step
print(f"Loaded {len(docs)} chunks")

[INFO] 2026-06-13 00:52:31,519 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-13 00:52:31,520 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-13 00:52:31,532 [RapidOCR] download_file.py:60: File exists and is valid: /Users/arokiabhavya/code/AIEngineering/.venv-1/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-13 00:52:31,532 [RapidOCR] main.py:50: Using /Users/arokiabhavya/code/AIEngineering/.venv-1/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-13 00:52:31,606 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-13 00:52:31,607 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-13 00:52:31,611 [RapidOCR] download_file.py:60: File exists and is valid: /Users/arokiabhavya/code/AIEngineering/.venv-1/lib/python3.14/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-13 00:52:31,612 [RapidOCR] main.py:50: Using /Users/arokiabhavy

Loaded 44 chunks


## Step 3 - Embedding

In [36]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8055.14it/s]


## Step 4 - Store into Qdrant (embedding happens automatically here)

In [37]:
from langchain_qdrant import QdrantVectorStore
vectorstore = QdrantVectorStore.from_documents(
    docs,
    embedding=embeddings,    # ← this is when vectors are actually computed
    location=":memory:",     # or a persistent path
    collection_name="hr_policies",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store ready!")

Vector store ready!


## Step 5 - LLM

In [38]:
import getpass
import os


from langchain_groq import ChatGroq
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")


GROQ_MODEL  = "openai/gpt-oss-safeguard-20b"
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=os.environ["GROQ_API_KEY"]
)
print("LLM ready!")

LLM ready!


## Step 6 -  RAG chain

In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a helpful HR assistant. Answer the question based only on the provided context.

Context: {context}

Question: {question}

Answer:
""")

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

RAG chain ready!


In [40]:
query = "How many casual leaves am I entitled to?"
response = rag_chain.invoke(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: How many casual leaves am I entitled to?

Response:
You’re entitled to **12 casual leaves per calendar year** (credited at one leave per month).
